# 02 — Fit the Jacobian lens and run the Table 3 validation battery

**Stage:** Proposal Stage 2. **Gate:** `Expected_Tables_and_Figures` §3, Table 3 — five
pass/fail checks on the *base model*, before AHN is involved at all. That document is
explicit: *"If either fails, the correct action is to fix the instrumentation, not to
report the RQ tables with a caveat."*

### Where the pilot stands

`fitted-jlens` fitted layer 18 from **one 106-character prompt** at `max_seq_len=64`.
The corpus-averaging step is the entire reason to prefer a J-lens over a logit lens —
the proposal says so in as many words: *"The averaging step is what separates
verbalizable content from content that merely happens to be verbalised in one context."*
A one-prompt map is a single-context Jacobian.

The evidence that it isn't working is already in `AHN_algoverse.ipynb` cell 24: decoding
the **full layer-18 residual stream** through J18 returns `<|endoftext|>`, `小镇`,
`县公安局`, `ABCDE`. The full residual at layer 18 should decode to something coherent —
that is the lens's easiest possible input. It is a fail, and it happened before any AHN
vector was decoded.

**Run Table 3 first. Nothing downstream is interpretable until it passes.**


In [ ]:
# --- bootstrap -------------------------------------------------------------------
# Upload `ahn_interp.py` next to this notebook (or anywhere up the tree).
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Upload it into this notebook's directory "
               "(Jupyter: Upload button, top right of the file browser).")
if _root not in sys.path:
    sys.path.insert(0, _root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)


In [ ]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
    cell            = "GatedDeltaNet",     # GatedDeltaNet | DeltaNet | Mamba2
    scale           = "3B",
    sliding_window  = 8064,                # proposal value; upstream eval uses 8064
    num_attn_sinks  = 128,                 # upstream eval default. NOT zero.
    attn_impl       = "flash_attention_2", # "eager" on T4/P100 (no Ampere -> no FA2)
    dtype           = "bfloat16",          # "float16" on T4/P100
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


In [ ]:
# The J-lens map is a property of the BACKBONE, not of the AHN checkpoint.
# Fit it once on Qwen2.5-3B-Instruct and reuse it for all three cells.
JCFG = dict(
    backbone      = "Qwen/Qwen2.5-3B-Instruct",
    source_layers = [9, 18, 27],   # stratified: early / middle / late
    n_prompts     = 500,           # proposal Table 3 stability check needs two x 500
    max_seq_len   = 256,
    skip_first    = 4,
    dim_batch     = 8,
    out_path      = os.path.join(CFG["results_dir"], "jlens_qwen25_3b.pt"),
)
print(json.dumps(JCFG, indent=2))


## The averaging corpus

Must be **strictly disjoint from every evaluation set** (proposal, Datasets section) so
the map cannot encode anything about the test items. Two disjoint halves so Table 3's
map-stability row can actually be computed.


In [ ]:
from datasets import load_dataset

def build_corpus(n, seed=ai.SEED, skip=0):
    """Generic English text, disjoint from RULER / LongBench / LV-Eval."""
    ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train", streaming=True)
    out = []
    for i, ex in enumerate(ds):
        if i < skip:
            continue
        t = ex["text"].strip()
        if len(t) > 400:
            out.append(t[:1500])
        if len(out) >= n:
            break
    return out

corpus_a = build_corpus(JCFG["n_prompts"], skip=0)
corpus_b = build_corpus(JCFG["n_prompts"], skip=20000)   # disjoint half
print(f"corpus A: {len(corpus_a)}   corpus B: {len(corpus_b)}")
print("overlap:", len(set(corpus_a) & set(corpus_b)), "(must be 0)")
assert not (set(corpus_a) & set(corpus_b))


## Fit

`jlens` needs `transformers >= 5.x`, which conflicts with the AHN repo's pin
(`transformers==4.51.0`). **Fit in a separate environment / separate session**, save the
`.pt`, and load it in notebook 04. That is why fitting and use are split across
notebooks — it is the version conflict Son hit, handled rather than worked around.

Budget check from the proposal: 15–20 GPU-hours expected, **abort RQ2 above ~40 h**.
The wall-clock printed below is Table 10 row 1. Record it.


In [ ]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.time()
btok = AutoTokenizer.from_pretrained(JCFG["backbone"])
bmodel = AutoModelForCausalLM.from_pretrained(
    JCFG["backbone"], torch_dtype=torch.bfloat16, device_map="auto"
).eval()

base = ai.ModelBundle(
    model=bmodel, tokenizer=btok, n_layers=bmodel.config.num_hidden_layers,
    hidden=bmodel.config.hidden_size, vocab=bmodel.lm_head.weight.shape[0],
    num_heads=bmodel.config.num_attention_heads,
    head_dim=getattr(bmodel.config, "head_dim",
                     bmodel.config.hidden_size // bmodel.config.num_attention_heads),
    sliding_window=None, num_attn_sinks=0, use_ahn_router=False, use_q_proj=False,
    use_normalized_l2=False, ahn_layers=[], ahn_impl="none", model_path=JCFG["backbone"],
)
print("backbone loaded", f"{time.time()-t0:.0f}s")


In [ ]:
t0 = time.time()
lens_a = ai.fit_jacobian_lens(
    base, corpus_a, JCFG["source_layers"],
    max_seq_len=JCFG["max_seq_len"], skip_first=JCFG["skip_first"],
    dim_batch=JCFG["dim_batch"],
)
fit_hours = (time.time() - t0) / 3600
lens_a.meta["fit_gpu_hours"] = fit_hours
lens_a.save(JCFG["out_path"])
print(f"fit corpus A in {fit_hours:.2f} GPU-hours -> {JCFG['out_path']}")
if fit_hours > 40 / len(JCFG["source_layers"]):
    print("! over the proposal's abort threshold — this is the RQ2 go/no-go signal")


## Table 3 — the five checks

| # | check | pass criterion |
|---|---|---|
| 1 | final-layer identity | lens at final layer reproduces the model's own next-token distribution, KL < 0.01 |
| 2 | logit-lens agreement | top-1 agreement with logit lens at mid layers ≥ 60% |
| 3 | known-fact recall | "The capital of France is" ranks " Paris" top-1 from ~layer 20 |
| 4 | map stability | two maps from disjoint 500-context corpora agree on top-10 ≥ 80% |
| 5 | map cost | wall-clock GPU-hours (gates the 7B decision, Table 10) |

Check 3 is the one to look at first — it is cheap, and it is the check the pilot's
layer-18 decode implicitly failed.


In [ ]:
import torch

@torch.no_grad()
def resid_at(model, tokenizer, prompt, layer, pos=-1):
    store = {}
    h = model.model.layers[layer].register_forward_hook(
        lambda m, i, o: store.__setitem__("h", (o[0] if isinstance(o, tuple) else o).detach())
    )
    try:
        ins = tokenizer(prompt, return_tensors="pt").to(model.device)
        model(**ins)
    finally:
        h.remove()
    return store["h"][0, pos].float()

T3 = {}

# --- check 3: known-fact recall ---------------------------------------------------
prompt = "The capital of France is"
paris = btok.encode(" Paris", add_special_tokens=False)[0]
rows = []
for L in JCFG["source_layers"]:
    v = resid_at(bmodel, btok, prompt, L)
    lg = ai.readout_logits(v, base, lens=lens_a, layer=L)
    rows.append({"layer": L, "rank_Paris": ai.token_rank(lg, paris),
                 "top5": [btok.decode([int(i)]) for i in lg.topk(5).indices]})
    print(rows[-1])
T3["known_fact_recall"] = {
    "rows": rows,
    "passed": any(r["rank_Paris"] == 0 for r in rows if r["layer"] >= 18),
}
print("\ncheck 3:", "PASS" if T3["known_fact_recall"]["passed"] else "FAIL")


In [ ]:
# --- check 2: agreement with the plain logit lens ---------------------------------
import numpy as np
probe_prompts = corpus_b[:40]
agree = {L: [] for L in JCFG["source_layers"]}
for p in probe_prompts:
    for L in JCFG["source_layers"]:
        v = resid_at(bmodel, btok, p[:800], L)
        j = ai.readout_logits(v, base, lens=lens_a, layer=L)
        l = ai.readout_logits(v, base, lens=None)
        agree[L].append(float(int(j.argmax()) == int(l.argmax())))
T3["logit_lens_agreement"] = {str(L): float(np.mean(a)) for L, a in agree.items()}
mid = JCFG["source_layers"][len(JCFG["source_layers"]) // 2]
T3["logit_lens_agreement"]["passed"] = bool(np.mean(agree[mid]) >= 0.60)
print(json.dumps(T3["logit_lens_agreement"], indent=2))


In [ ]:
# --- check 1: final-layer identity ------------------------------------------------
# The lens at the final layer should be (near) the identity: transporting the last
# residual and decoding must reproduce the model's own next-token distribution.
final_layer = base.n_layers - 1
if final_layer in lens_a.jacobians:
    p_txt = corpus_b[0][:600]
    ins = btok(p_txt, return_tensors="pt").to(bmodel.device)
    with torch.no_grad():
        true_logits = bmodel(**ins).logits[0, -1].float()
    v = resid_at(bmodel, btok, p_txt, final_layer)
    lens_logits = ai.readout_logits(v, base, lens=lens_a, layer=final_layer)
    P = torch.softmax(true_logits, -1); Q = torch.softmax(lens_logits, -1)
    kl = float((P * (P.clamp_min(1e-12).log() - Q.clamp_min(1e-12).log())).sum())
    T3["final_layer_identity"] = {"kl": kl, "passed": kl < 0.01}
else:
    T3["final_layer_identity"] = {
        "skipped": True,
        "note": f"layer {final_layer} not in source_layers; add it to run this check",
    }
print(json.dumps(T3["final_layer_identity"], indent=2))


In [ ]:
# --- check 4: map stability across disjoint corpora --------------------------------
# The expensive one. Halve n_prompts for both halves if compute is tight, and say so.
lens_b = ai.fit_jacobian_lens(
    base, corpus_b, JCFG["source_layers"],
    max_seq_len=JCFG["max_seq_len"], skip_first=JCFG["skip_first"],
    dim_batch=JCFG["dim_batch"],
)
overlaps = {}
for L in JCFG["source_layers"]:
    hits = []
    for p in corpus_b[:30]:
        v = resid_at(bmodel, btok, p[:800], L)
        ta = set(int(i) for i in ai.readout_logits(v, base, lens=lens_a, layer=L).topk(10).indices)
        tb = set(int(i) for i in ai.readout_logits(v, base, lens=lens_b, layer=L).topk(10).indices)
        hits.append(len(ta & tb) / 10.0)
    overlaps[str(L)] = float(np.mean(hits))
T3["map_stability"] = {**overlaps, "passed": bool(min(overlaps.values()) >= 0.80)}
print(json.dumps(T3["map_stability"], indent=2))


In [ ]:
T3["map_cost_gpu_hours"] = fit_hours
T3["config"] = JCFG
required = ["known_fact_recall", "logit_lens_agreement", "map_stability"]
T3["TABLE_3_PASSED"] = all(T3[k].get("passed", False) for k in required)

ai.save_json(T3, "02_table3_jlens_validation.json")
print(json.dumps({k: (v.get("passed") if isinstance(v, dict) else v)
                  for k, v in T3.items()}, indent=2))
print("\nTABLE 3:", "PASS" if T3["TABLE_3_PASSED"] else "FAIL — fix the lens, do not proceed")


### If Table 3 fails

In order of likelihood:

1. **Corpus too small or too short.** Raise `n_prompts` and `max_seq_len` before
   anything else — this is the pilot's failure mode.
2. **`skip_first` too small.** Early positions are dominated by the BOS/template
   prefix and drag the average.
3. **Missing final norm.** `ai.readout_logits(..., apply_final_norm=True)` handles it;
   the pilot's `vec @ unembed.T` did not.
4. **Wrong layer convention.** `jlens` may index the residual *before* vs *after* a
   block differently from a `register_forward_hook` on `model.model.layers[L]`. Test by
   sweeping ±1 and seeing which makes check 3 pass.

Report the outcome to Gautam either way. The proposal has a **hard go/no-go at Week 8**:
if the lens cannot be made to work, RQ2 and RQ3 are dropped and the paper falls back to
RQ1. Knowing that in Week 6 is worth more than a working lens in Week 10.
